In [1]:
!git clone https://github.com/SLDGroup/EMCAD.git
%cd EMCAD


Cloning into 'EMCAD'...
remote: Enumerating objects: 267, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 267 (delta 71), reused 55 (delta 54), pack-reused 185 (from 3)
Receiving objects: 100% (267/267), 2.05 MiB | 18.77 MiB/s, done.
Resolving deltas: 100% (126/126), done.
/content/EMCAD


In [2]:
import torch
import timm
import cv2
import albumentations
import einops

print("All imports successful!")

All imports successful!


In [3]:


!pip install -q gdown

!gdown 1wvmw8DVyDKr5sOAFn5zUpfhbK4Vxjze4

Downloading...
From (original): https://drive.google.com/uc?id=1wvmw8DVyDKr5sOAFn5zUpfhbK4Vxjze4
From (redirected): https://drive.google.com/uc?id=1wvmw8DVyDKr5sOAFn5zUpfhbK4Vxjze4&confirm=t&uuid=292e35f7-37db-4064-b4c7-f4fd31a01201
To: /content/EMCAD/synapse.zip
100% 766M/766M [00:08<00:00, 86.4MB/s]


In [4]:
!mkdir -p ./data/synapse
!unzip -q synapse.zip -d ./data/synapse/

In [5]:
# =========================================================
# INSTALL
# =========================================================

!pip install timm -q

# =========================================================
# IMPORTS
# =========================================================

import os
import glob
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

import timm

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

# =========================================================
# DEVICE
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# =========================================================
# DATASET
# =========================================================

class SynapseDataset(Dataset):

    def __init__(self, data_dir):

        self.files = glob.glob(data_dir + "/*.npz")

    def __len__(self):

        return len(self.files)

    def __getitem__(self, idx):

        data = np.load(self.files[idx])

        image = data["image"]
        mask = data["label"]

        image = torch.tensor(image, dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.long)

        # [H,W] -> [1,H,W]
        image = image.unsqueeze(0)

        return image, mask


train_dataset = SynapseDataset(
    "./data/synapse/synapse/train_npz_new"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Dataset Size:", len(train_dataset))

# =========================================================
# CAB
# =========================================================

class CAB(nn.Module):

    def __init__(self, channels, reduction=16):

        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))

        attention = avg_out + max_out
        attention = self.sigmoid(attention)

        return x * attention

# =========================================================
# SAB
# =========================================================

class SAB(nn.Module):

    def __init__(self, kernel_size=7):

        super().__init__()

        padding = kernel_size // 2

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size,
            padding=padding,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)

        attn = torch.cat([avg, mx], dim=1)

        attn = self.sigmoid(self.conv(attn))

        return x * attn

# =========================================================
# MSDC
# =========================================================

class MSDC(nn.Module):

    def __init__(self, channels, kernels=(1,3,5), stride=1):

        super().__init__()

        self.branches = nn.ModuleList()

        for k in kernels:

            self.branches.append(
                nn.Sequential(
                    nn.Conv2d(
                        channels,
                        channels,
                        kernel_size=k,
                        stride=stride,
                        padding=k//2,
                        groups=channels,
                        bias=False
                    ),
                    nn.BatchNorm2d(channels),
                    nn.ReLU(inplace=True)
                )
            )

    def forward(self, x):

        outputs = []

        for branch in self.branches:

            outputs.append(branch(x))

        return outputs

# =========================================================
# MSCB
# =========================================================

class MSCB(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1,
        kernels=(1,3,5),
        expansion=2
    ):

        super().__init__()

        self.use_skip = (
            stride == 1 and
            in_channels == out_channels
        )

        hidden = in_channels * expansion

        self.expand = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True)
        )

        self.msdc = MSDC(hidden, kernels, stride)

        self.project = nn.Sequential(
            nn.Conv2d(
                hidden * len(kernels),
                out_channels,
                1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):

        identity = x

        x = self.expand(x)

        x = torch.cat(self.msdc(x), dim=1)

        x = self.project(x)

        if self.use_skip:

            x = x + identity

        return x

# =========================================================
# EUCB
# =========================================================

class EUCB(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.up = nn.Upsample(
            scale_factor=2,
            mode='bilinear',
            align_corners=False
        )

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                in_channels,
                3,
                padding=1,
                groups=in_channels,
                bias=False
            ),

            nn.BatchNorm2d(in_channels),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                in_channels,
                out_channels,
                1,
                bias=False
            )
        )

    def forward(self, x):

        x = self.up(x)

        x = self.block(x)

        return x

# =========================================================
# LGAG
# =========================================================

class LGAG(nn.Module):

    def __init__(self, F_g, F_l, F_int):

        super().__init__()

        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )

        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):

        g1 = self.W_g(g)
        x1 = self.W_x(x)

        psi = self.relu(g1 + x1)

        psi = self.psi(psi)

        return x * psi

# =========================================================
# PVTv2 ENCODER
# =========================================================

class PVTv2Encoder(nn.Module):

    def __init__(self):

        super().__init__()

        # grayscale -> RGB
        self.input_proj = nn.Conv2d(1, 3, kernel_size=1)

        # pretrained PVTv2 backbone
        self.backbone = timm.create_model(
            'pvt_v2_b2',
            pretrained=True,
            features_only=True
        )

    def forward(self, x):

        x = self.input_proj(x)

        features = self.backbone(x)

        """
        features[0] -> [B, 64, H/4,  W/4]
        features[1] -> [B,128, H/8,  W/8]
        features[2] -> [B,320, H/16, W/16]
        features[3] -> [B,512, H/32, W/32]
        """

        x1 = features[0]
        x2 = features[1]
        x3 = features[2]
        x4 = features[3]

        return x4, [x3, x2, x1]

# =========================================================
# EMCAD DECODER
# =========================================================

class EMCAD(nn.Module):

    def __init__(self):

        super().__init__()

        self.cab = CAB(512)
        self.sab = SAB()

        self.u3 = EUCB(512, 320)
        self.g3 = LGAG(320, 320, 160)
        self.m3 = MSCB(320, 320)

        self.u2 = EUCB(320, 128)
        self.g2 = LGAG(128, 128, 64)
        self.m2 = MSCB(128, 128)

        self.u1 = EUCB(128, 64)
        self.g1 = LGAG(64, 64, 32)
        self.m1 = MSCB(64, 64)

    def forward(self, x, skips):

        x = self.cab(x)
        x = self.sab(x)

        d4 = x

        d3 = self.u3(d4)
        d3 = self.g3(d3, skips[0])
        d3 = self.m3(d3)

        d2 = self.u2(d3)
        d2 = self.g2(d2, skips[1])
        d2 = self.m2(d2)

        d1 = self.u1(d2)
        d1 = self.g1(d1, skips[2])
        d1 = self.m1(d1)

        return d4, d3, d2, d1

# =========================================================
# FULL MODEL
# =========================================================

class EMCAD_Model(nn.Module):

    def __init__(self, num_classes=14):

        super().__init__()

        self.encoder = PVTv2Encoder()

        self.decoder = EMCAD()

        self.head_d4 = nn.Conv2d(512, num_classes, 1)
        self.head_d3 = nn.Conv2d(320, num_classes, 1)
        self.head_d2 = nn.Conv2d(128, num_classes, 1)
        self.head_d1 = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):

        x, skips = self.encoder(x)

        d4, d3, d2, d1 = self.decoder(x, skips)

        p4 = self.head_d4(d4)
        p3 = self.head_d3(d3)
        p2 = self.head_d2(d2)
        p1 = self.head_d1(d1)

        return [p4, p3, p2, p1]

# =========================================================
# MODEL
# =========================================================

model = EMCAD_Model(num_classes=14).to(device)

print("Model Loaded Successfully!")


Device: cpu
Dataset Size: 2211


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/101M [00:00<?, ?B/s]

Model Loaded Successfully!


In [ ]:

# =========================================================
# LOSS
# =========================================================

def dice_loss(pred, target, num_classes=14):

    pred = torch.softmax(pred, dim=1)

    target_onehot = F.one_hot(
        target,
        num_classes=num_classes
    ).permute(0,3,1,2).float()

    target_onehot = target_onehot.to(pred.device)

    intersection = (
        pred * target_onehot
    ).sum(dim=(2,3))

    union = (
        pred.sum(dim=(2,3)) +
        target_onehot.sum(dim=(2,3))
    )

    dice = (
        2 * intersection + 1e-6
    ) / (
        union + 1e-6
    )

    return 1 - dice.mean()

def total_loss(pred, target):

    ce = F.cross_entropy(pred, target)

    d = dice_loss(pred, target)

    return ce + d

# =========================================================
# OPTIMIZER
# =========================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

# =========================================================
# TEST FORWARD PASS
# =========================================================

img, mask = next(iter(train_loader))

img = img.to(device)

outputs = model(img)

for i, out in enumerate(outputs):

    print(f"Output {i} Shape:", out.shape)

# =========================================================
# GOOGLE DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# CHECKPOINT PATHS
# =========================================================

epochs = 100

checkpoint_dir = "/content/drive/MyDrive/"

latest_path = os.path.join(
    checkpoint_dir,
    "emcad_latest.pth"
)

best_path = os.path.join(
    checkpoint_dir,
    "emcad_best.pth"
)

start_epoch = 0
best_loss = float("inf")

# =========================================================
# RESUME TRAINING
# =========================================================

if os.path.exists(latest_path):

    checkpoint = torch.load(
        latest_path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    start_epoch = checkpoint["epoch"] + 1

    best_loss = checkpoint.get(
        "best_loss",
        float("inf")
    )

    print(f"Resuming From Epoch {start_epoch}")

# =========================================================
# TRAINING LOOP
# =========================================================

model.train()

for epoch in range(start_epoch, epochs):

    total_train_loss = 0.0

    loop = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}"
    )

    for img, mask in loop:

        img = img.to(device)
        mask = mask.to(device)

        outputs = model(img)

        loss = 0.0

        for out in outputs:

            out = F.interpolate(
                out,
                size=mask.shape[-2:],
                mode='bilinear',
                align_corners=False
            )

            loss += total_loss(out, mask)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_train_loss += loss.item()

        loop.set_postfix(
            loss=loss.item()
        )

    avg_loss = total_train_loss / len(train_loader)

    print(f"\nEpoch {epoch+1} Avg Loss: {avg_loss:.4f}")

    # =====================================================
    # SAVE LATEST
    # =====================================================

    torch.save({

        "epoch": epoch,

        "model_state_dict": model.state_dict(),

        "optimizer_state_dict": optimizer.state_dict(),

        "best_loss": best_loss

    }, latest_path)

    # =====================================================
    # SAVE PER EPOCH
    # =====================================================

    torch.save({

        "epoch": epoch,

        "model_state_dict": model.state_dict(),

        "optimizer_state_dict": optimizer.state_dict(),

        "best_loss": best_loss

    },

    os.path.join(
        checkpoint_dir,
        f"emcad_epoch_{epoch}.pth"
    ))

    # =====================================================
    # SAVE BEST MODEL
    # =====================================================

    if avg_loss < best_loss:

        best_loss = avg_loss

        torch.save({

            "epoch": epoch,

            "model_state_dict": model.state_dict(),

            "optimizer_state_dict": optimizer.state_dict(),

            "best_loss": best_loss

        }, best_path)

        print(
            f"New BEST Model Saved "
            f"At Epoch {epoch+1} "
            f"(Loss={best_loss:.4f})"
        )

print("Training Finished Successfully!")